# Data analysis

## Criação do dataset, validação e limpeza do mesmo

In [69]:
import os
import pandas as pd

# Recursive function to find all CSV files in a directory and its subdirectories
def find_csvs(directory):
    csv_files = []
    for item in os.listdir(directory):
        full_path = os.path.join(directory, item)
        if os.path.isdir(full_path):
            if item == "processed":
                continue
            csv_files.extend(find_csvs(full_path))
        elif item.endswith(".csv"):
            csv_files.append(full_path)
    return csv_files

all_files = find_csvs(os.path.join("..", "src", "data"))
print(f"Found {len(all_files)} CSV files\n")

df = pd.concat((pd.read_csv(f) for f in all_files), ignore_index=True)
print(f"Combined DataFrame shape: {df.shape}\n")

# Printing all variables in the DataFrame
print("DataFrame columns:")
for column in df.columns:
    print(column)

Found 24 CSV files

Combined DataFrame shape: (13577024, 19)

DataFrame columns:
YEAR
QUARTER
MONTH
DAY_OF_WEEK
OP_CARRIER_AIRLINE_ID
OP_CARRIER_FL_NUM
ORIGIN
ORIGIN_STATE_ABR
DEST
DEST_STATE_ABR
CRS_DEP_TIME
DEP_DELAY
CRS_ARR_TIME
ARR_DELAY_NEW
CANCELLED
DIVERTED
CRS_ELAPSED_TIME
DISTANCE
DISTANCE_GROUP


In [70]:
# Verifying how many unique flights each month and year have making sure that the data is consistent and there are no missing months or years
flight_counts = df.groupby(['YEAR', 'MONTH']).size().reset_index(name='flight_count')
print("\n")
for index, row in flight_counts.iterrows():
    print(f"Year: {row['YEAR']}, Month: {row['MONTH']}, Flight Count: {row['flight_count']}")



Year: 2022, Month: 1, Flight Count: 537902
Year: 2022, Month: 2, Flight Count: 495713
Year: 2022, Month: 3, Flight Count: 564853
Year: 2022, Month: 4, Flight Count: 556502
Year: 2022, Month: 5, Flight Count: 578819
Year: 2022, Month: 6, Flight Count: 577283
Year: 2022, Month: 7, Flight Count: 594957
Year: 2022, Month: 8, Flight Count: 589810
Year: 2022, Month: 9, Flight Count: 557494
Year: 2022, Month: 10, Flight Count: 572287
Year: 2022, Month: 11, Flight Count: 546410
Year: 2022, Month: 12, Flight Count: 557095
Year: 2023, Month: 1, Flight Count: 538837
Year: 2023, Month: 2, Flight Count: 502749
Year: 2023, Month: 3, Flight Count: 580322
Year: 2023, Month: 4, Flight Count: 561441
Year: 2023, Month: 5, Flight Count: 579958
Year: 2023, Month: 6, Flight Count: 577262
Year: 2023, Month: 7, Flight Count: 601866
Year: 2023, Month: 8, Flight Count: 602987
Year: 2023, Month: 9, Flight Count: 569338
Year: 2023, Month: 10, Flight Count: 598968
Year: 2023, Month: 11, Flight Count: 563777
Year

In [71]:
# Verifying each variable misssing values and its percentage of missing values as well as its data type
for column in df.columns:
    missing_values = df[column].isnull().sum()
    percentage_missing = (missing_values / len(df)) * 100
    data_type = df[column].dtype

    if missing_values > 0:
        print(f"Column: {column}, Missing Values: {missing_values}, Percentage: {percentage_missing}, Data Type: {data_type}")

Column: DEP_DELAY, Missing Values: 261877, Percentage: 1.928824755704932, Data Type: float64
Column: ARR_DELAY_NEW, Missing Values: 301609, Percentage: 2.221466206438171, Data Type: float64
Column: CRS_ELAPSED_TIME, Missing Values: 5, Percentage: 3.6826921717159814e-05, Data Type: float64


In [72]:
# Checking for duplicate rows in the DataFrame
duplicate_rows = df[df.duplicated()]
print(f"\nNumber of duplicate rows: {len(duplicate_rows)}")

# Eliminating duplicates if any are found
before_deduplication_size = df.shape[0]
df = df.drop_duplicates()
after_deduplication_size = df.shape[0]
print(f"Removed {before_deduplication_size - after_deduplication_size} duplicate rows. New DataFrame shape: {df.shape}")

# Cleaning the DataFrame for model development by removing canceled or diverted flights and considering only flights with a valid recorded arrival delay (ARR_DELAY)
before_cleaning_size = df.shape[0]
df = df[(df['CANCELLED'] != 1) & (df['DIVERTED'] != 1)]
removed_canceled_or_diverted_count = before_cleaning_size - df.shape[0]
print(f"\nRemoved {removed_canceled_or_diverted_count} canceled or diverted flights.")
print(f"DataFrame size after removing canceled or diverted flights: {df.shape[0]}")

before_arr_filter = df.shape[0]
df = df[df['ARR_DELAY_NEW'].notnull()]
print(f"\nRemoved {before_arr_filter - df.shape[0]} rows with missing ARR_DELAY_NEW.")
print(f"DataFrame size after keeping only flights with valid ARR_DELAY_NEW: {df.shape[0]}")

# Removing rows with 9 as the day of week (DAY_OF_WEEK)
before_day_filter = df.shape[0]
df = df[df['DAY_OF_WEEK'] != 9]
print(f"\nRemoved {before_day_filter - df.shape[0]} rows with DAY_OF_WEEK equal to 9.")

# Checking for any remaining missing values in the DataFrame
for column in df.columns:
    missing_values = df[column].isnull().sum()
    if missing_values > 0:
        print(f"Column: {column} still has {missing_values} missing values.")


Number of duplicate rows: 513402
Removed 513402 duplicate rows. New DataFrame shape: (13063622, 19)

Removed 286573 canceled or diverted flights.
DataFrame size after removing canceled or diverted flights: 12777049

Removed 2 rows with missing ARR_DELAY_NEW.
DataFrame size after keeping only flights with valid ARR_DELAY_NEW: 12777047

Removed 0 rows with DAY_OF_WEEK equal to 9.


In [73]:
# Date and temporal features
print(f"YEAR range: {df['YEAR'].min()} - {df['YEAR'].max()}")
print(f"MONTH range: {df['MONTH'].min()} - {df['MONTH'].max()}")
print(f"DAY_OF_WEEK range: {df['DAY_OF_WEEK'].min()} - {df['DAY_OF_WEEK'].max()} (expected 1-7)")
print(f"QUARTER range: {df['QUARTER'].min()} - {df['QUARTER'].max()}")

# Time features (HHMM format)
print(f"\nCRS_DEP_TIME range: {df['CRS_DEP_TIME'].min()} - {df['CRS_DEP_TIME'].max()}")
print(f"CRS_ARR_TIME range: {df['CRS_ARR_TIME'].min()} - {df['CRS_ARR_TIME'].max()}")
invalid_dep = df[(df['CRS_DEP_TIME'] % 100) >= 60].shape[0]
invalid_arr = df[(df['CRS_ARR_TIME'] % 100) >= 60].shape[0]
print(f"Invalid DEP_TIME minutes (>=60): {invalid_dep}")
print(f"Invalid ARR_TIME minutes (>=60): {invalid_arr}")

# Distance and duration (should be positive)
print(f"\nDISTANCE range: {df['DISTANCE'].min()} - {df['DISTANCE'].max()}")
print(f"CRS_ELAPSED_TIME range: {df['CRS_ELAPSED_TIME'].min()} - {df['CRS_ELAPSED_TIME'].max()}")
print(f"Negative DISTANCE: {(df['DISTANCE'] < 0).sum()}")
print(f"Negative CRS_ELAPSED_TIME: {(df['CRS_ELAPSED_TIME'] < 0).sum()}")

YEAR range: 2022 - 2023
MONTH range: 1 - 12
DAY_OF_WEEK range: 1 - 7 (expected 1-7)
QUARTER range: 1 - 4

CRS_DEP_TIME range: 1 - 2359
CRS_ARR_TIME range: 1 - 2400
Invalid DEP_TIME minutes (>=60): 0
Invalid ARR_TIME minutes (>=60): 0

DISTANCE range: 21.0 - 5095.0
CRS_ELAPSED_TIME range: -85.0 - 1162.0
Negative DISTANCE: 0
Negative CRS_ELAPSED_TIME: 5


In [74]:
# Creating a new column 'ARR_DELAY_CATEGORY' based on the 'ARR_DELAY_NEW' values
def categorize_delay(delay):
    if delay < 15:
        return 'on_time'
    elif delay < 60:
        return 'moderate_delay'
    else:
        return 'severe_delay'

df['ARR_DELAY_CATEGORY'] = df['ARR_DELAY_NEW'].apply(categorize_delay)

In [75]:
# Deleting the columns that are not needed for the model development
columns_to_drop = ['DEP_DELAY', 'CANCELLED', 'DIVERTED', 'ARR_DELAY_NEW', 'DISTANCE_GROUP']
df.drop(columns=columns_to_drop, inplace=True)
df.reset_index(drop=True, inplace=True)

# New DataFrame shape
print(f"\nFinal DataFrame shape after dropping unnecessary columns: {df.shape}")


Final DataFrame shape after dropping unnecessary columns: (12777047, 15)


In [76]:
# How many unique flight numbers?
print(f"OP_CARRIER_FL_NUM unique values: {df['OP_CARRIER_FL_NUM'].nunique()}")
print(f"Total rows: {len(df):,}")

# Clean 5 rows with negative CRS_ELAPSED_TIME
before = len(df)

# Print them
df = df[df['CRS_ELAPSED_TIME'] > 0]
print(f"\nRemoved {before - len(df)} rows with negative CRS_ELAPSED_TIME")

OP_CARRIER_FL_NUM unique values: 6851
Total rows: 12,777,047

Removed 5 rows with negative CRS_ELAPSED_TIME


In [ ]:
# Printing the columns of the DataFrame
print("\n DataFrame columns:")
for column in df.columns:
    print(column)


Final DataFrame columns:
YEAR
QUARTER
MONTH
DAY_OF_WEEK
OP_CARRIER_AIRLINE_ID
OP_CARRIER_FL_NUM
ORIGIN
ORIGIN_STATE_ABR
DEST
DEST_STATE_ABR
CRS_DEP_TIME
CRS_ARR_TIME
CRS_ELAPSED_TIME
DISTANCE
ARR_DELAY_CATEGORY


## EDA

### Distribuição do target

In [78]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Show both the number and percentage of flights in each class.
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = ['#00F3FF', '#66CCCC', '#334455']
target_counts = df['ARR_DELAY_CATEGORY'].value_counts().reindex(order)
target_percentages = target_counts / target_counts.sum() * 100

fig = go.Figure(go.Bar(
    x=order,
    y=target_counts,
    marker_color=colors,
    text=[f'{count:,} ({percentage:.2f}%)' for count, percentage in zip(target_counts, target_percentages)],
    textposition='outside',
    cliponaxis=False
))
fig.update_layout(
    title='Flight Arrival Delay Severity',
    xaxis_title='Delay category',
    yaxis_title='Number of flights',
    height=500,
    width=800,
    showlegend=False
)
fig.update_yaxes(range=[0, target_counts.max() * 1.15])
fig.show()


#### Podemos observar que as classes estão despropocionais, o que faz sentido visto.

### Distribuição do target por mês

In [79]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Calculating the monthly percentage of each delay category
monthly_pct = pd.crosstab(
    index=[df['YEAR'], df['MONTH']],
    columns=df['ARR_DELAY_CATEGORY'],
    normalize='index'
) * 100

monthly_pct = monthly_pct.sort_index()

month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
labels = [f"{month_names[m]} {y}" for y, m in monthly_pct.index]

fig = go.Figure()
for category in order:
    fig.add_trace(go.Scatter(
        x=labels,
        y=monthly_pct[category],
        mode='lines+markers',
        name=category,
        line=dict(color=colors[category], width=2),
        marker=dict(size=6)
    ))

fig.update_layout(
    title='Target Distribution Over Time',
    xaxis_title='Month',
    yaxis_title='Percentage (%)',
    height=500,
    width=1100,
    xaxis=dict(tickangle=45, dtick=1),
    legend_title='Delay Category'
)
fig.show()

#### Não revela nenhum padrão relevante mas foi útil

### Distribuição do target por Top 10 Companhias Aeras com mais voos

In [80]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Filter for the top 10 airlines by flight volume
top10_airlines = df['OP_CARRIER_AIRLINE_ID'].value_counts().head(10).index
df_top10 = df[df['OP_CARRIER_AIRLINE_ID'].isin(top10_airlines)]

# Calculate the percentage of each delay category per airline
airline_pct = pd.crosstab(
    index=df_top10['OP_CARRIER_AIRLINE_ID'],
    columns=df_top10['ARR_DELAY_CATEGORY'],
    normalize='index'
) * 100

# Sort airlines by flight volume (most flights on top)
volume_order = df_top10['OP_CARRIER_AIRLINE_ID'].value_counts().index
airline_pct = airline_pct.reindex(volume_order)

fig = go.Figure()
for category in order:
    fig.add_trace(go.Bar(
        y=airline_pct.index.astype(str),
        x=airline_pct[category],
        name=category,
        marker_color=colors[category],
        orientation='h'
    ))

fig.update_layout(
    barmode='stack',
    title='Target Distribution — Top 10 Airlines by Flight Volume',
    xaxis_title='Percentage (%)',
    yaxis_title='Airline ID',
    height=500,
    width=900,
    legend_title='Delay Category'
)
fig.show()

#### Dá para perceber que certos aeroportos têm pior desempenho que outros

### Distribuição do target por hora de partida

In [81]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Extract departure hour from HHMM format
df['DEP_HOUR'] = df['CRS_DEP_TIME'] // 100

# Calculate the percentage of each delay category per hour
hour_pct = pd.crosstab(
    index=df['DEP_HOUR'],
    columns=df['ARR_DELAY_CATEGORY'],
    normalize='index'
) * 100

# Already sorted by hour naturally (0, 1, 2, ..., 23)

fig = go.Figure()
for category in order:
    fig.add_trace(go.Bar(
        x=hour_pct.index,
        y=hour_pct[category],
        name=category,
        marker_color=colors[category]
    ))

fig.update_layout(
    barmode='stack',
    title='Target Distribution by Departure Hour',
    xaxis_title='Departure Hour',
    yaxis_title='Percentage (%)',
    height=500,
    width=900,
    xaxis=dict(dtick=1),
    legend_title='Delay Category'
)
fig.show()

# Clean up temporary column
df.drop(columns=['DEP_HOUR'], inplace=True)

#### Das 18 até às 4 existe um maior atraso

### Distribuição de target por distância

In [82]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Create distance groups
dist_bins = [0, 250, 500, 750, 1000, 1500, 2000, 3000, 6000]
dist_labels = ['0-250', '250-500', '500-750', '750-1k', '1k-1.5k', '1.5k-2k', '2k-3k', '3k+']
df['DIST_BIN'] = pd.cut(df['DISTANCE'], bins=dist_bins, labels=dist_labels)

# Calculate the percentage of each delay category per distance group
dist_pct = pd.crosstab(
    index=df['DIST_BIN'],
    columns=df['ARR_DELAY_CATEGORY'],
    normalize='index'
) * 100

fig = go.Figure()
for category in order:
    fig.add_trace(go.Bar(
        x=dist_pct.index.astype(str),
        y=dist_pct[category],
        name=category,
        marker_color=colors[category]
    ))

fig.update_layout(
    barmode='stack',
    title='Target Distribution by Distance Group',
    xaxis_title='Distance (miles)',
    yaxis_title='Percentage (%)',
    height=500,
    width=900,
    legend_title='Delay Category'
)
fig.show()

# Clean up temporary column
df.drop(columns=['DIST_BIN'], inplace=True)

#### Distância por si não é muito descriminativo mas pode ajudar

### Visualização com 2 features

#### Top 10 companhia aerea de acordo com a distancia

In [83]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Filter for top 10 airlines by flight volume
top10_airlines = df['OP_CARRIER_AIRLINE_ID'].value_counts().head(10).index
df_top10 = df[df['OP_CARRIER_AIRLINE_ID'].isin(top10_airlines)]

# Create distance groups
dist_bins = [0, 250, 500, 750, 1000, 1500, 2000, 3000, 6000]
dist_labels = ['0-250', '250-500', '500-750', '750-1k', '1k-1.5k', '1.5k-2k', '2k-3k', '3k+']
df_top10 = df_top10.copy()
df_top10['DIST_BIN'] = pd.cut(df_top10['DISTANCE'], bins=dist_bins, labels=dist_labels)

# Calculate total delay rate (moderate + severe) per airline and distance group
total = df_top10.groupby(['OP_CARRIER_AIRLINE_ID', 'DIST_BIN'], observed=True).size()
delayed = df_top10[df_top10['ARR_DELAY_CATEGORY'] != 'on_time'].groupby(['OP_CARRIER_AIRLINE_ID', 'DIST_BIN'], observed=True).size()
delay_rate = (delayed / total * 100).unstack()

# Order airlines by flight volume (most flights on top)
volume_order = df_top10['OP_CARRIER_AIRLINE_ID'].value_counts().index
delay_rate = delay_rate.reindex(volume_order)

# Replace NaN with "-" in display text, show grey for missing cells
display_text = delay_rate.copy()
display_text = display_text.map(lambda x: f'{x:.1f}%' if pd.notna(x) else '-')

fig = go.Figure(go.Heatmap(
    z=delay_rate.values,
    x=[str(c) for c in delay_rate.columns],
    y=delay_rate.index.astype(str),
    text=display_text.values,
    texttemplate='%{text}',
    colorscale=[[0, '#00F3FF'], [0.5, '#66CCCC'], [1, '#334455']],
    colorbar=dict(title='Delay %'),
    hoverongaps=False
))

fig.update_layout(
    title='Total Delay Rate (%) — Airline × Distance (Top 10 Airlines)',
    xaxis_title='Distance (miles)',
    yaxis_title='Airline ID',
    height=500,
    width=900
)
fig.show()

#### Claramente existe companhias que têm pior perfomance como antes visto mas a distância também influencia o delay

#### Top 10 companhia aerea de acordo com a hora de partida

In [84]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Filter for top 10 airlines by flight volume
top10_airlines = df['OP_CARRIER_AIRLINE_ID'].value_counts().head(10).index
df_top10 = df[df['OP_CARRIER_AIRLINE_ID'].isin(top10_airlines)].copy()

# Extract departure hour from HHMM format
df_top10['DEP_HOUR'] = df_top10['CRS_DEP_TIME'] // 100

# Calculate total delay rate (moderate + severe) per airline and hour
total = df_top10.groupby(['OP_CARRIER_AIRLINE_ID', 'DEP_HOUR'], observed=True).size()
delayed = df_top10[df_top10['ARR_DELAY_CATEGORY'] != 'on_time'].groupby(['OP_CARRIER_AIRLINE_ID', 'DEP_HOUR'], observed=True).size()
delay_rate = (delayed / total * 100).unstack()

# Order airlines by flight volume (most flights on top)
volume_order = df_top10['OP_CARRIER_AIRLINE_ID'].value_counts().index
delay_rate = delay_rate.reindex(volume_order)

# Replace NaN with "-" in display text
display_text = delay_rate.map(lambda x: f'{x:.1f}%' if pd.notna(x) else '-')

fig = go.Figure(go.Heatmap(
    z=delay_rate.values,
    x=[str(c) for c in delay_rate.columns],
    y=delay_rate.index.astype(str),
    text=display_text.values,
    texttemplate='%{text}',
    colorscale=[[0, '#00F3FF'], [0.5, '#66CCCC'], [1, '#334455']],
    colorbar=dict(title='Delay %'),
    hoverongaps=False
))

fig.update_layout(
    title='Total Delay Rate (%) — Airline × Departure Hour (Top 10 Airlines)',
    xaxis_title='Departure Hour',
    yaxis_title='Airline ID',
    height=500,
    width=900,
    xaxis=dict(dtick=1)
)
fig.show()

#### Departure time é uma feature importante

### Hora por dia da Semana

In [85]:
order = ['on_time', 'moderate_delay', 'severe_delay']
colors = {'on_time': '#00F3FF', 'moderate_delay': '#66CCCC', 'severe_delay': '#334455'}

# Extract departure hour
df['DEP_HOUR'] = df['CRS_DEP_TIME'] // 100

day_map = {1:'Mon', 2:'Tue', 3:'Wed', 4:'Thu', 5:'Fri', 6:'Sat', 7:'Sun'}

# Calculate total delay rate per hour and day of week
total = df.groupby(['DEP_HOUR', 'DAY_OF_WEEK'], observed=True).size()
delayed = df[df['ARR_DELAY_CATEGORY'] != 'on_time'].groupby(['DEP_HOUR', 'DAY_OF_WEEK'], observed=True).size()
delay_rate = (delayed / total * 100).unstack()

# Rename columns to day names
delay_rate.columns = [day_map[d] for d in delay_rate.columns]

# Replace NaN with "-" in display text
display_text = delay_rate.map(lambda x: f'{x:.1f}%' if pd.notna(x) else '-')

fig = go.Figure(go.Heatmap(
    z=delay_rate.values,
    x=[str(c) for c in delay_rate.columns],
    y=delay_rate.index.astype(str),
    text=display_text.values,
    texttemplate='%{text}',
    colorscale=[[0, '#00F3FF'], [0.5, '#66CCCC'], [1, '#334455']],
    colorbar=dict(title='Delay %'),
    hoverongaps=False
))

fig.update_layout(
    title='Total Delay Rate (%) — Departure Hour × Day of Week',
    xaxis_title='Day of Week',
    yaxis_title='Departure Hour',
    height=600,
    width=800,
    yaxis=dict(dtick=1)
)
fig.show()

# Clean up temporary column
df.drop(columns=['DEP_HOUR'], inplace=True)

#### Hora de partida discrimina muito mais que o dia da semana, mas o dia acrescenta informação, especialmente a diferença entre meio da semana e sexta/domingo.

#### Ver correlação entre distância e tempo esperado de viagem

In [86]:
# Pearson correlation between Distance and Flight Duration
corr = df[['DISTANCE', 'CRS_ELAPSED_TIME']].corr().iloc[0, 1]
print(f"Pearson r (Distance vs Flight Duration): {corr:.4f}")

# Scatter plot with a 5000-row sample for performance
scatter_sample = df.sample(5000, random_state=42)

fig = go.Figure(go.Scatter(
    x=scatter_sample['DISTANCE'],
    y=scatter_sample['CRS_ELAPSED_TIME'],
    mode='markers',
    marker=dict(color='#00F3FF', opacity=0.3)
))

fig.update_layout(
    title=f'Distance vs Flight Duration (r = {corr:.3f})',
    xaxis_title='Distance (miles)',
    yaxis_title='Flight Duration (min)',
    height=450,
    width=650
)
fig.show()

Pearson r (Distance vs Flight Duration): 0.9830


#### Muito redudantes como se estava à espera, uma delas pode ser removida

### Estado de origem com hora de partida

In [87]:
# Calculate delay rate per airport as origin and destination
origin_total = df.groupby('ORIGIN').size()
origin_delayed = df[df['ARR_DELAY_CATEGORY'] != 'on_time'].groupby('ORIGIN').size()
origin_rate = (origin_delayed / origin_total * 100)

dest_total = df.groupby('DEST').size()
dest_delayed = df[df['ARR_DELAY_CATEGORY'] != 'on_time'].groupby('DEST').size()
dest_rate = (dest_delayed / dest_total * 100)

# Merge and keep only airports with at least 1000 flights in each role
compare = pd.DataFrame({'As Origin': origin_rate, 'As Destination': dest_rate}).dropna()
valid = compare.index[
    (origin_total.reindex(compare.index) >= 1000) &
    (dest_total.reindex(compare.index) >= 1000)
]
compare = compare.loc[valid]

# Select top 15 airports by combined flight volume
top15 = (origin_total + dest_total).reindex(valid).nlargest(15).index
compare = compare.loc[top15]

# Correlation between origin and destination delay rates
corr = compare['As Origin'].corr(compare['As Destination'])
print(f"Pearson r (Origin vs Dest delay rate): {corr:.4f}")

fig = go.Figure()
fig.add_trace(go.Bar(
    x=compare.index.astype(str),
    y=compare['As Origin'],
    name='As Origin',
    marker_color='#2ecc71'
))
fig.add_trace(go.Bar(
    x=compare.index.astype(str),
    y=compare['As Destination'],
    name='As Destination',
    marker_color='#e74c3c'
))

fig.update_layout(
    barmode='group',
    title=f'Delay Rate by Airport Role — Top 15 Airports (r = {corr:.3f})',
    xaxis_title='Airport',
    yaxis_title='Delay Rate (%)',
    height=500,
    width=900
)
fig.show()

Pearson r (Origin vs Dest delay rate): 0.7899


#### Alguns estados têm valores diferentes logo ambas podem ser úteis

## Discussão e análise de features e consecutiva seleão final

In [88]:
# Printing all variables in the DataFrame again
print("DataFrame columns:")
for column in df.columns:
    print(column)

DataFrame columns:
YEAR
QUARTER
MONTH
DAY_OF_WEEK
OP_CARRIER_AIRLINE_ID
OP_CARRIER_FL_NUM
ORIGIN
ORIGIN_STATE_ABR
DEST
DEST_STATE_ABR
CRS_DEP_TIME
CRS_ARR_TIME
CRS_ELAPSED_TIME
DISTANCE
ARR_DELAY_CATEGORY


#### Kept as raw values:
- DEPARTURE_TIME / ARRIVAL_TIME kept in HHMM format, maybe extract hour (// 100)
- DAY_OF_WEEK kept as 1 7, derive IS_WEEKEND flag if needed
- ORIGIN_AIRPORT / DEST_AIRPORT high cardinality (~350 unique), handle encoding
- ORIGIN_STATE / DEST_STATE coarser than airport, evaluate with feature importance
- ARRIVAL_TIME   possibly derivable from DEPARTURE_TIME + DISTANCE, evaluate with feature importance


In [89]:
# QUARTER: redundant with MONTH
# OP_CARRIER_FL_NUM too many unique values
# CRS_ELAPSED_TIME too much correlation with DISTANCE

cols_redundant = ['QUARTER', 'OP_CARRIER_FL_NUM', 'CRS_ELAPSED_TIME']
df.drop(columns=cols_redundant, inplace=True)

# Renaming columns for clarity
df.rename(columns={
    'OP_CARRIER_AIRLINE_ID': 'AIRLINE_ID',
    'ORIGIN': 'ORIGIN_AIRPORT',
    'DEST': 'DEST_AIRPORT',
    'ORIGIN_STATE_ABR': 'ORIGIN_STATE',
    'DEST_STATE_ABR': 'DEST_STATE',
    'CRS_DEP_TIME': 'DEPARTURE_TIME',
    'CRS_ARR_TIME': 'ARRIVAL_TIME'
}, inplace=True)

for column in df.columns:
    print(column)

print("\nFinal DataFrame shape:", df.shape)

# Save the final cleaned dataset
output_dir = os.path.join("..", "src", "data", "processed")
os.makedirs(output_dir, exist_ok=True)
output_file_path = os.path.join(output_dir, "cleaned_flights_data.csv")
df.to_csv(output_file_path, index=False)
print(f"\nFinal dataset saved to {output_file_path}")

YEAR
MONTH
DAY_OF_WEEK
AIRLINE_ID
ORIGIN_AIRPORT
ORIGIN_STATE
DEST_AIRPORT
DEST_STATE
DEPARTURE_TIME
ARRIVAL_TIME
DISTANCE
ARR_DELAY_CATEGORY

Final DataFrame shape: (12777042, 12)

Final dataset saved to ..\src\data\processed\cleaned_flights_data.csv
